# Chapter 6: Ensemble Learning and Random Forests

Ensemble Learning is based on the idea of the "Wisdom of the Crowd." Instead of relying on one highly complex model, we train a group of different models (called an ensemble) and combine their predictions. 

## 1. The Magic of the Crowd (Law of Large Numbers)
The first concept explores why combining models works so well, using a simple coin-toss analogy.

Imagine you have a slightly biased coin that comes up heads 51% of the time and tails 49% of the time. 
* If you toss it just a few times, you might get more tails than heads.
* However, if you toss it **10,000 times**, the mathematical *Law of Large Numbers* guarantees that the ratio of heads will get closer and closer to 51%. The probability of getting a majority of heads after 10,000 tosses is incredibly high (close to 97%).

**How this applies to Machine Learning:**
If you build 1,000 different, independent Machine Learning models, and each model is only correct 51% of the time (barely better than random guessing), combining their majority vote can result in an ensemble that is highly accurate (e.g., 97% accurate)! A group of **weak learners** can mathematically form a **strong learner**.


## 2. Hard Voting Classifiers
A `VotingClassifier` is the simplest way to create an ensemble in Scikit-Learn. It trains several different algorithms on the exact same dataset and lets them vote on the final prediction.

In the code example, we combine three entirely different models:
1. `LogisticRegression` (A linear model)
2. `RandomForestClassifier` (A tree-based model)
3. `SVC` (Support Vector Classifier)

**How "Hard Voting" works:**
When a new data point arrives, each of the three models makes an independent prediction (e.g., Model 1 says "Class A", Model 2 says "Class A", Model 3 says "Class B"). The Voting Classifier simply counts the votes. Since "Class A" got 2 out of 3 votes, the ensemble predicts "Class A".

```python
# Individual model scores:
# Logistic Regression: 86.4%
# Random Forest: 89.6%
# SVC: 89.6%

# The Hard Voting Ensemble score:
print(voting_clf.score(X_test, y_test)) 
# Output: 91.2%
```
**Conclusion:** The combined ensemble (91.2%) beats even the best individual model in the group!


## 3. Soft Voting Classifiers
Hard voting is very democratic, but it ignores *how confident* a model is. If Model 1 is 99% sure it's "Class B", but Models 2 and 3 are only 51% sure it's "Class A", Hard Voting will blindly choose "Class A".

**Soft Voting** fixes this by averaging the predicted probabilities of all the models. 
To use it, all models in the ensemble must be able to estimate class probabilities (for `SVC`, you must explicitly set `probability=True`), and you change the voting strategy to `voting="soft"`.

```python
voting_clf.voting = "soft"
voting_clf.fit(X_train, y_train)

print(voting_clf.score(X_test, y_test))
# Output: 92.0%
```
**Conclusion:** Soft Voting gives more weight to highly confident votes. It almost always achieves higher performance than Hard Voting, pushing our accuracy from 91.2% to 92.0%.

## 4. Bagging and Pasting
If we want to build a powerful ensemble using the *same* algorithm (e.g., 500 Decision Trees), we need a way to make them diverse so they don't all make the same mistakes. We achieve this by training each predictor on a different random subset of the training set.

There are two main ways to draw these random subsets:
*   **Bagging (Bootstrap Aggregating):** Sampling is performed *with replacement*. A single instance can be sampled multiple times for the same predictor.
*   **Pasting:** Sampling is performed *without replacement*. A single instance can be sampled at most once for a given predictor.

Bagging is generally preferred because it introduces more diversity (due to replacement), which results in a model with a slightly higher bias but a significantly lower variance.

### Bagging in Scikit-Learn
Scikit-Learn offers a simple API for this: the `BaggingClassifier` (or `BaggingRegressor` for continuous targets).

```python
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Train an ensemble of 500 Decision Trees
# Each tree is trained on 100 randomly sampled instances (with replacement by default)
# n_jobs=-1 tells Scikit-Learn to use all available CPU cores for parallel training
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=100, n_jobs=-1, random_state=42
)
bag_clf.fit(X_train, y_train)
```

### Visualizing the Power of Bagging
If we compare the decision boundary of a single Decision Tree versus a Bagging ensemble of 500 trees:
*   **Single Decision Tree:** The boundary is highly irregular, jagged, and attempts to memorize outliers (High Variance / Overfitting).
*   **Bagging Ensemble:** The boundary is much smoother and more generalized. The ensemble makes fewer errors on the training data and will perform significantly better on unseen data.

## 5. Out-of-Bag (OOB) Evaluation

When using Bagging (sampling *with replacement*), some instances in the dataset will be sampled several times for any given predictor, while others might not be sampled at all. 

By default, a `BaggingClassifier` samples $m$ training instances with replacement. The mathematical probability of an instance *not* being picked is $(1 - 1/m)^m$. As $m$ (the size of the dataset) grows large, this ratio approaches $e^{-1} \approx 0.37$. 
This means that roughly **63%** of the training instances are sampled for each predictor, and the remaining **37%** are never seen by that predictor. These unseen instances are called **Out-of-Bag (OOB)** instances.

### The "Free" Validation Set
Since a predictor never sees its OOB instances during training, it can be evaluated on them. We can evaluate the entire ensemble by averaging the OOB evaluations of each predictor. 
This is a massive advantage because we get a reliable validation score without needing to waste training data on a separate validation set!

In Scikit-Learn, you can enable this automatic evaluation by setting `oob_score=True`:

```python
# Train a Bagging ensemble and automatically evaluate it on OOB instances
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    oob_score=True, n_jobs=-1, random_state=42
)
bag_clf.fit(X_train, y_train)

# Check the internal OOB evaluation score
print(bag_clf.oob_score_)
# Output: 0.896 (roughly 89.6% accuracy)

# We can verify this is reliable by testing it on a completely separate test set:
from sklearn.metrics import accuracy_score
y_pred = bag_clf.predict(X_test)
print(accuracy_score(y_test, y_pred))
# Output: 0.920
```
The OOB score (`89.6%`) is usually a very good and conservative estimate of the final test score (`92.0%`).

## 6. Random Forests

A Random Forest is essentially an ensemble of Decision Trees, usually trained via the bagging method. However, it introduces an extra layer of randomness to make the trees even more diverse.

If we only use standard Bagging, and there is one highly dominant feature in our dataset, almost all the trees will use that same dominant feature at their root node. The trees will still end up being highly correlated (similar to each other).

### The Random Forest Trick (Feature Sampling)
To force the trees to be completely different, a Random Forest restricts what features a tree can see. At each node, when the tree is looking for the best possible split, it is **only allowed to search within a random subset of features** (typically the square root of the total number of features). 

This forces the trees to explore hidden patterns in less dominant features, resulting in greater tree diversity, which ultimately yields a better and more robust ensemble.

### Implementation in Scikit-Learn
You can use the optimized `RandomForestClassifier` directly. As shown below, it is mathematically and programmatically equivalent to building a `BaggingClassifier` of `DecisionTreeClassifier`s where `max_features="sqrt"`.

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# Method 1: Using the dedicated Random Forest class
rnd_clf = RandomForestClassifier(
    n_estimators=500, max_leaf_nodes=16, 
    n_jobs=-1, random_state=42
)
rnd_clf.fit(X_train, y_train)
y_pred_rf = rnd_clf.predict(X_test)

# Method 2: Building the exact same model using Bagging
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(max_features="sqrt", max_leaf_nodes=16),
    n_estimators=500, n_jobs=-1, random_state=42
)
bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)

# Verify they produce the exact same predictions:
print(np.all(y_pred_bag == y_pred_rf))
# Output: True
```

## 7. Feature Importance

Another incredible quality of Random Forests is that they make it easy to measure the relative importance of each feature. 

Scikit-Learn measures a feature's importance by looking at how much the tree nodes that use that feature reduce impurity on average (across all trees in the forest). It computes this score automatically for each feature after training, and scales the results so that the sum of all importances is equal to 1.

You can access these scores using the `feature_importances_` variable:

```python
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, random_state=42)
rnd_clf.fit(iris.data, iris.target)

for name, score in zip(iris.feature_names, rnd_clf.feature_importances_):
    print(name, score)
    
# Output Example:
# sepal length (cm) 0.11
# sepal width (cm) 0.02
# petal length (cm) 0.44  <-- High importance
# petal width (cm) 0.42   <-- High importance
```

### Visualizing Importance (MNIST Example)
If we train a Random Forest on image data (like the MNIST handwritten digits), we can reshape the `feature_importances_` array back into the shape of an image (28x28 pixels) and plot it as a heatmap. 

The resulting visualization shows that the pixels at the edges of the image have almost zero importance (black), because numbers are rarely drawn there. The pixels in the center of the image have the highest importance (yellow/white), as these are the crucial pixels needed to differentiate between different digits.

## 8. Boosting (Sequential Ensemble)

Unlike Bagging, where models are trained independently and in parallel, **Boosting** is an ensemble method where models are trained **sequentially**. The core idea is simple: each new model in the sequence should focus on correcting the mistakes made by its predecessor.

### AdaBoost (Adaptive Boosting)
One of the most popular boosting algorithms is AdaBoost. Here is how it works under the hood:
1.  **First Predictor:** A base classifier (like a shallow Decision Tree) is trained and makes predictions on the training set.
2.  **Weight Update:** The algorithm identifies the instances that the first model misclassified and **boosts (increases) their relative weights**.
3.  **Second Predictor:** The next classifier is trained on the same dataset, but because of the updated weights, it pays much more attention to the instances the first model got wrong.
4.  **Ensemble Prediction:** This sequential process continues. To make a final prediction, all models vote, but their votes are weighted based on their individual accuracy on the training set.

### Implementation in Scikit-Learn
*Note: In newer versions of Scikit-Learn, the `algorithm` parameter is deprecated/removed because `SAMME.R` (which relies on class probabilities) is usually the default.*

```python
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Train an AdaBoost ensemble using 30 shallow Decision Trees (Decision Stumps)
ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=30,
    learning_rate=0.5, random_state=42
)
ada_clf.fit(X_train, y_train)
```
*Visual Impact:* As shown in Figure 6-8, as the algorithm progresses or as the `learning_rate` increases, the decision boundaries become increasingly fitted to the difficult, previously misclassified instances.

## 9. Gradient Boosting

While AdaBoost tweaks the *instance weights* at every step, **Gradient Boosting** tries to fit the new predictor directly to the **residual errors** made by the previous predictor.

### The Manual Math
If we use Decision Trees as our base predictors, this is called Gradient Tree Boosting. Here is how it works sequentially:
1. Train `tree_reg1` on the original data `(X, y)`.
2. Calculate the residual errors: `y2 = y - tree_reg1.predict(X)`.
3. Train `tree_reg2` on the errors `(X, y2)`.
4. Calculate new residual errors: `y3 = y2 - tree_reg2.predict(X)`.
5. Train `tree_reg3` on `(X, y3)`.

The final ensemble prediction is simply the sum of the predictions from all trees: 
`y_pred = tree_reg1.predict(X) + tree_reg2.predict(X) + tree_reg3.predict(X)`

### Gradient Boosting in Scikit-Learn
Scikit-Learn provides `GradientBoostingRegressor` (and Classifier) to do this automatically. 

**The Shrinkage Technique (Learning Rate):**
The `learning_rate` hyperparameter scales the contribution of each tree. 
* If you set it low (e.g., `0.05`), you will need many more trees in the ensemble to fit the training set, but the predictions will usually generalize much better. 
* To find the optimal number of trees without overfitting, we can use **Early Stopping**. By setting `n_iter_no_change=10`, the algorithm stops training automatically if the validation score doesn't improve for 10 consecutive iterations.

```python
from sklearn.ensemble import GradientBoostingRegressor

# A robust model using a low learning rate and early stopping
gbrt_best = GradientBoostingRegressor(
    max_depth=2, learning_rate=0.05, n_estimators=500,
    n_iter_no_change=10, random_state=42
)
gbrt_best.fit(X, y)
print("Optimal number of trees:", gbrt_best.n_estimators_)
# Output: 53 (It stopped early, long before reaching 500!)
```

## 10. Histogram-Based Gradient Boosting (HGB)

Standard Gradient Boosting works well, but it scales terribly on massive datasets. The bottleneck is that the Decision Tree algorithm must evaluate every single unique value of a continuous feature to find the optimal split point. For millions of rows, this requires massive computation ($O(n \times m \log m)$).

**HGB (HistGradientBoostingRegressor/Classifier)** solves this problem by binning (discretizing) continuous input features into integer bins (typically 255 bins maximum) before training begins. 

### Why is HGB so powerful?
1.  **Blazing Fast:** Instead of evaluating millions of split points, the tree only needs to evaluate a maximum of 255 bins per feature. The time complexity drops to $O(b \times m)$ where $b$ is the number of bins.
2.  **Categorical Features:** HGB natively supports categorical features. You don't even need to apply One-Hot Encoding!
3.  **Missing Values:** It has built-in support for missing values (`NaN`), automatically learning whether missing data should go to the left or right child node.

*Note: Binning causes a slight loss of precision (it acts as a regularizer), which helps prevent overfitting but might cause slight underfitting on very small, clean datasets.*

```python
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

# Creating a pipeline to handle categorical features directly using HGBR
hgb_reg = make_pipeline(
    make_column_transformer(
        (OrdinalEncoder(), ["ocean_proximity"]),
        remainder="passthrough"
    ),
    HistGradientBoostingRegressor(categorical_features=[0], random_state=42)
)

# Fit the model (it will train incredibly fast even on massive datasets)
hgb_reg.fit(housing, housing_labels)
```

# 11. Stacking (Stacked Generalization)

Stacking is an advanced ensemble technique. Instead of using trivial functions (like hard voting or simple averaging) to combine the predictions of all the base models, stacking uses a **meta-learner** (also called a blender) to learn how to combine them effectively.

## How Stacking Works
1. **Base Predictors:** Several different base models (e.g., Logistic Regression, Random Forest, SVC) make their independent predictions.
2. **Out-of-Fold Predictions:** To prevent data leakage, the meta-learner is trained on predictions made by the base predictors using cross-validation (`cv` parameter).
3. **The Meta-Learner:** The predictions from the base models are fed as *new features* into a final model (the meta-learner), which learns which base models are more reliable and outputs the final prediction.

## Implementation in Scikit-Learn
Scikit-Learn provides the `StackingClassifier` (and `StackingRegressor`) to build stacked ensembles easily.

```python
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Define the base estimators and the final meta-learner
stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    final_estimator=RandomForestClassifier(random_state=43),
    cv=5  # Number of cross-validation folds for out-of-fold predictions
)

# Train the stacking ensemble
stacking_clf.fit(X_train, y_train)

# Evaluate the final score
print(stacking_clf.score(X_test, y_test))
# Output: 0.928
```

# Chapter 6 Exercises & Solutions

Here is a summary of the conceptual exercises for Ensemble Learning. These questions test our intuition about how different ensemble methods work under the hood.

### Q1: Combining 5 different models with 95% precision
**Question:** If you have trained five different models on the exact same training data, and they all achieve 95% precision, is there any chance that you can combine these models to get better results? 
**Answer:** **Yes!** You can combine them into a voting ensemble. This leverages the "wisdom of the crowd." However, it works best if the models are fundamentally different (e.g., an SVM, a Decision Tree, and a Logistic Regression). If they are different, they will likely make different types of errors. Combining them will average out those errors and improve the overall result.

### Q2: Hard vs. Soft Voting
**Question:** What is the difference between hard and soft voting classifiers?
**Answer:** 
*   **Hard Voting:** Strictly counts the final categorical votes of each classifier and picks the majority class (Absolute Democracy).
*   **Soft Voting:** Computes the average of the *estimated class probabilities* across all models and picks the highest one. It gives more weight to highly confident models and generally performs better. (Requires all models to have `probability=True`).

### Q3: Parallelization (Speeding up training)
**Question:** Is it possible to speed up training of a bagging ensemble by distributing it across multiple servers? What about pasting ensembles, boosting ensembles, random forests, or stacking ensembles?
**Answer:**
*   **Bagging, Pasting, and Random Forests:** **Yes.** Each tree is built completely independently of the others, so you can train them simultaneously on multiple servers (Parallel).
*   **Boosting:** **No.** Boosting builds models sequentially. Tree #2 *needs* the errors from Tree #1 to learn. They cannot be parallelized across servers.
*   **Stacking:** **Yes and No.** All predictors within a single layer can be trained in parallel. However, Layer 2 cannot start training until Layer 1 is completely finished.

### Q4: Out-of-Bag (OOB) Benefit
**Question:** What is the benefit of out-of-bag evaluation?
**Answer:** In Bagging, each predictor only sees about 63% of the training data. The remaining 37% (OOB) are unseen. We can use these OOB instances as a "free" validation set to evaluate the model's performance. This saves us from having to split our precious training data to create a separate validation set.

### Q5: Extra-Trees vs. Random Forests
**Question:** What makes Extra-Trees ensembles more random than regular Random Forests? How does this help? Are they slower or faster?
**Answer:** 
*   **Randomness:** While Random Forests use random subsets of *features*, Extra-Trees go one step further: they also use random *thresholds* for splitting, rather than searching for the absolute best possible threshold like a normal Decision Tree.
*   **Benefit:** This extreme randomness acts as a strong **regularization** technique, which is very helpful if a standard Random Forest is overfitting the training data.
*   **Speed:** Because they skip the heavy mathematical computation of finding the "best" split threshold, they are **much faster to train**. However, prediction speed is the same.

### Q6: AdaBoost Underfitting
**Question:** If your AdaBoost ensemble underfits the training data, which hyperparameters should you tweak?
**Answer:** Underfitting means the model is too simple and isn't learning enough. You can:
1.  Increase the number of estimators (`n_estimators`).
2.  Reduce the regularization of the base estimator (e.g., increase `max_depth` of the base Decision Tree).
3.  Slightly increase the `learning_rate` so the model takes stronger steps to correct errors.

### Q7: Gradient Boosting Overfitting
**Question:** If your Gradient Boosting ensemble overfits the training set, should you increase or decrease the learning rate?
**Answer:** Overfitting means the model is too complex and is memorizing the noise. You should **decrease the learning rate** (this acts as a brake, forcing the model to learn slower). You should also use **Early Stopping** to halt training before the model starts memorizing the data.

# Chapter 6 Summary: Ensemble Learning and Random Forests

In this chapter, we explored the concept of **"The Wisdom of the Crowd."** The core idea of Ensemble Learning is that if you aggregate the predictions of a group of predictors (such as classifiers or regressors), you will often get better predictions than with the best individual predictor.

Here is a quick recap of the major techniques we covered:

### 1. Voting Classifiers
*   **Hard Voting:** The ensemble makes a prediction by taking a simple majority vote of all the base classifiers.
*   **Soft Voting:** The ensemble computes the average of the estimated class probabilities output by all classifiers and predicts the class with the highest probability. It generally performs better than hard voting because it gives more weight to highly confident models.

### 2. Bagging and Pasting
Instead of using different training algorithms, we can use the *same* algorithm but train them on different random subsets of the training set. These models are trained **in parallel**.
*   **Bagging:** Sampling *with* replacement. Introduces more diversity and is generally preferred.
*   **Pasting:** Sampling *without* replacement.
*   **Out-of-Bag (OOB) Evaluation:** In Bagging, about 37% of the instances are never seen by each predictor. We can use these unseen instances as a "free" validation set, eliminating the need for a separate validation split.

### 3. Random Forests
A Random Forest is an ensemble of Decision Trees, generally trained via the bagging method.
*   **Feature Randomness:** To force the trees to be even more diverse, Random Forests only consider a random subset of features (rather than all features) when searching for the best split at each node.
*   **Feature Importance:** Random Forests make it easy to measure the relative importance of each feature by calculating how much the tree nodes that use that feature reduce impurity across all trees.

### 4. Boosting
Boosting is an ensemble method where predictors are trained **sequentially**, with each new model trying to correct the mistakes of its predecessor.
*   **AdaBoost:** Each new predictor pays more attention to the training instances that the previous predictor misclassified by increasing their relative weights.
*   **Gradient Boosting:** Instead of tweaking instance weights, each new predictor tries to fit the *residual errors* made by the previous predictor.
*   **HGBR (Histogram-Based Gradient Boosting):** An optimized version of Gradient Boosting for massive datasets. It bins continuous features into integer bins (max 255), making it blazing fast and natively supporting categorical features and missing values.

### 5. Stacking (Stacked Generalization)
Instead of using simple functions (like voting or averaging) to aggregate the predictions of all predictors, Stacking trains a **Meta-Learner** (or blender) to figure out how to combine them. The base models predict on the data, and their outputs are used as *new features* to train the final Meta-Learner.